# Entity Detection & Completeness Evaluation Pipeline

**Purpose**: Evaluate the quality of entity extraction by measuring detection accuracy, similarity scores, and completeness.

**Key workflow**:
- Load ground truth entity labels from previous extraction
- Compare extracted entities against OCR text using LLM-based detection
- Calculate fuzzy similarity scores for entity matching
- Assess entity completeness and relevance
- Generate comprehensive evaluation metrics and reports

## Import Dependencies & Environment

- Load environment variables for Azure credentials
- Import Azure OpenAI client for evaluation tasks
- Bring in `rapidfuzz` for fuzzy string matching
- Import `json` and `re` for data handling and text cleaning
- Use `DefaultAzureCredential` for Azure AD authentication

In [1]:
import os
import json
from rapidfuzz import fuzz
import re
from dotenv import load_dotenv
load_dotenv()

from openai import AzureOpenAI
import re
from azure.identity import DefaultAzureCredential

## Configure Azure OpenAI Client Factory

- Define reusable function to create authenticated Azure OpenAI client
- Leverage `DefaultAzureCredential` for automatic token acquisition
- Wrap token refresh logic in `token_provider` closure
- Returns client instance ready for evaluation prompts

In [2]:
def get_llm_client_instance(
    azure_endpoint: str,
    api_version: str,
    azure_deployment: str,
):
    credential = DefaultAzureCredential()

    # Token provider for Azure OpenAI
    def token_provider():
        token = credential.get_token(
            "https://cognitiveservices.azure.com/.default"
        )
        return token.token

    client = AzureOpenAI(
        azure_endpoint=azure_endpoint,
        api_version=api_version,
        azure_deployment=azure_deployment,
        azure_ad_token_provider=token_provider,
    )

    return client

## Load Ground Truth Entity Labels

- Read previously extracted entity labels from `labels.json`
- Labels contain entity names, values, and Chinese translations per page
- Display loaded data for verification

In [3]:
labelling_output_path = "/Users/anishganguli/Documents/Projects/HSBC/PoC/HSBC_IWPB_UW/document_extraction/src/data/results/labels.json"

with open(labelling_output_path, "r") as json_file:
    labels_data = json.load(json_file)
    print("Labels data:", labels_data)

Labels data: [{'page_number': 1, 'entity_presence': {'Applicant Name': True}, 'entity_value': {'Applicant Name': 'Ms LOK WING CHING'}, 'chinese_entity_value': {'Applicant Name': '林詠菁'}}, {'page_number': 2, 'entity_presence': {'Job Title of Applicant': True}, 'entity_value': {'Job Title of Applicant': 'DIRECTOR'}, 'chinese_entity_value': {}}, {'page_number': 4, 'entity_presence': {'Business Registration Number of Employer': True}, 'entity_value': {'Business Registration Number of Employer': '21893829'}, 'chinese_entity_value': {'Business Registration Number of Employer': '21893829'}}, {'page_number': 8, 'entity_presence': {'Height of Applicant': True, 'Weight of Applicant': True}, 'entity_value': {'Height of Applicant': '174 cm', 'Weight of Applicant': '77 kg'}, 'chinese_entity_value': {'Height of Applicant': '身高 174 公分', 'Weight of Applicant': '體重 77 公斤'}}]


## Restructure Labels for Page-Indexed Access

- Transform labels into page-centric structure
- For each page:
  - Extract page number
  - Collect entity name, value, and Chinese value triplets
- Build `pages_index` dictionary mapping page numbers to entity lists
- Enables efficient lookup during evaluation

In [4]:
pages_label_data = []

for item in labels_data:
    page_number = item.get("page_number") 
    # page_number = 1

    entities = []

    for name, value in item["entity_value"].items():
        entities.append({
            "entity_name": name,
            "entity_value": value,
            "chinese_entity_value": item["chinese_entity_value"].get(name)
        })

    pages_label_data.append({
        "page_number": page_number,
        "entities": entities
    })

pages_index = {
    entry["page_number"]: entry["entities"]
    for entry in pages_label_data
}


## Define Entity Detection Evaluation Prompt

**Goal**: Verify whether extracted entity values appear in OCR text

**Process**:
- Accepts ground truth entities with names and values
- Compares against provided extracted text
- Returns JSON indicating detection status (true/false) for each entity

**Output format**: Array of entity values with boolean detection flags

In [5]:
system_entity_detection_evaluation_prompt = """
You are an expert document understanding AI model. Your task is to evaluate the accuracy of entity detection in a given text based on provided ground truth data.

### GOAL
Your goal is to compare the detected entities against the ground truth entities and provide if the detected entities are present in the given text.

### INPUT
You will be provided with:
- The text extracted from a document.
- A list of ground truth entities, each with an entity name and its corresponding value. The list might contain only one entity as well.

### INSTRUCTIONS
- Read the provided text very carefully and understand its content.
- For each ground truth entity, check if the entity and the correspomding value is present in the provided text.
- If the entity value is found in the text, mark it as "true", otherwise mark it as "false".

### OUTPUT
Provide your evaluation in the following JSON format:
{
  "eval": [
            {
              "entity_value": "value_of_entity1",
              "is_detected": true_or_false
            },
            {
              "entity_value": "value_of_entity2",
              "is_detected": true_or_false
            },
            ...
  ]
}
"""

## Instantiate Authenticated Client

- Create Azure OpenAI client using factory function
- Pull endpoint, API version, and deployment from environment
- Uses token-based authentication for secure access

In [6]:
llm_client = get_llm_client_instance(
    azure_endpoint = os.getenv("GPT_4_1_API_ENDPOINT"),
    api_version = os.getenv("GPT_4_1_API_VERSION"),
    azure_deployment = os.getenv("GPT_4_1_API_DEPLOYMENT"),
)

## Run Entity Detection Evaluation

- Iterate through pages (1, 2, 4, 8)
- **For each page**:
  - Load corresponding OCR text file
  - Format entities list from ground truth labels
  - Send text + entities to Azure OpenAI for detection validation
  - Parse JSON response and clean markdown artifacts
  - Attach page number to results
- Collect all evaluations in `final_eval` list

In [7]:
page_numbers = [1, 2, 4, 8]
final_eval = []

for page in page_numbers:
    adi_text_path = f"/Users/anishganguli/Documents/Projects/HSBC/PoC/HSBC_IWPB_UW/document_extraction/src/data/results/adi_text_page_{page}.txt"

    with open(adi_text_path, "r") as f:
        adi_text = f.read()

    entities_block = "\n".join(
        f"- Entity Name: {e['entity_name']}, Entity Value: {e['entity_value']}"
        for e in pages_index[page]
    )

    messages = [
        {
            "role": "system",
            "content": system_entity_detection_evaluation_prompt,
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        f"Entities:\n{entities_block}\n\n"
                        f"Text:\n{adi_text}"
                    )
                }
            ]
        }
    ]

    response = llm_client.chat.completions.create(
        model=os.getenv("GPT_4_1_API_DEPLOYMENT"),
        messages=messages,
        temperature=0.0,
    )

    eval_result = json.loads(
        response.choices[0].message.content
        .replace("```json", "")
        .replace("```", "")
    )["eval"]

    final_eval.append({
        "page_number": page,
        "entity_detection_evaluation": eval_result
    })


## Define OCR Text Cleaning Function

- Remove HTML tags using regex
- Collapse multiple whitespace characters into single spaces
- Strip leading/trailing whitespace
- Prepares text for accurate fuzzy matching

In [8]:
def clean_ocr_text(text):
    # remove HTML tags
    text = re.sub(r"<.*?>", " ", text)
    # collapse multiple spaces
    text = re.sub(r"\s+", " ", text)
    return text.strip()

## Calculate Fuzzy Similarity Scores

- Process each page and its entities
- **For each entity**:
  - Load and clean OCR text
  - Compute token-set similarity ratio between entity value and full text
  - Normalize score to 0-1 range
  - Store entity name, value, and similarity score
- Build `fuzzy_scores` dictionary indexed by page number

In [9]:
fuzzy_scores = {}

for page_number in page_numbers:
    adi_text_path = f"/Users/anishganguli/Documents/Projects/HSBC/PoC/HSBC_IWPB_UW/document_extraction/src/data/results/adi_text_page_{page_number}.txt"

    with open(adi_text_path, "r") as f:
        adi_text = f.read()

    # store scores per entity
    fuzzy_scores[page_number] = []

    for entity in pages_index[page_number]:
        entity_value = entity["entity_value"]
        clean_text = clean_ocr_text(adi_text)
        similarity_score = fuzz.token_set_ratio(entity_value, clean_text) / 100.0
        fuzzy_scores[page_number].append({
            "entity_name": entity["entity_name"],
            "entity_value": entity_value,
            "entity_match_similarity": similarity_score
        })


## Merge Detection & Similarity Data

- Combine LLM-based detection results with fuzzy similarity scores
- **For each page**:
  - Retrieve fuzzy scores and detection evaluation
  - Load cleaned OCR text
  - Merge entity metadata with detection status and similarity score
- Produce unified `all_eval` list with comprehensive entity evaluation data

In [10]:
adi_text_path = "/Users/anishganguli/Documents/Projects/HSBC/PoC/HSBC_IWPB_UW/document_extraction/src/data/results/adi_text_page_{}.txt"

all_eval = []

for page_eval in final_eval: 
    page_number = page_eval["page_number"]
    
    entities_fuzzy = fuzzy_scores.get(page_number, [])
    
    with open(adi_text_path.format(page_number), "r") as f:
        adi_text_raw = f.read()
    adi_text = clean_ocr_text(adi_text_raw)
    
    for i, entity in enumerate(entities_fuzzy):
        is_detected = page_eval["entity_detection_evaluation"][i]["is_detected"]
        
        all_eval.append({
            "page_number": page_number,
            "entity_name": entity["entity_name"],
            "entity_value": entity["entity_value"],
            "adi_text": adi_text,
            "is_detected": is_detected,
            "entity_match_similarity": entity["entity_match_similarity"]
        })


## Compute Entity Detection Rate

- Count total entities and successfully detected entities
- Calculate detection rate as percentage
- Display overall detection accuracy metric

In [11]:
total_entities = 0
detected_entities = 0

for page in all_eval:
    if page["is_detected"]:
        detected_entities += 1
    total_entities += 1

entity_detection_rate = detected_entities / total_entities
print("Entity Detection Rate:", 100 * entity_detection_rate, "%")

Entity Detection Rate: 100.0 %


## Calculate Entity Detection Similarity Score

- Sum similarity scores across all entities
- Divide by entity count to get average
- Display average fuzzy matching accuracy as percentage

In [12]:
total_score = 0.0
num_entities = len(all_eval)

for entity in all_eval:
    total_score += entity["entity_match_similarity"]

avg_fuzzy_score = total_score / num_entities if num_entities > 0 else 0.0

print(f"Entity Detection Similarity across all entities: {100 * avg_fuzzy_score:.2f}%")


Entity Detection Similarity across all entities: 100.00%


## Define Completeness Score Calculator

- Returns 0.0 if entity is not relevant
- Returns 1.0 if entity is complete
- For incomplete entities:
  - Count missing information items (capped at normalizer value)
  - Calculate score as 1.0 minus (missing_count / normalizer)
- Default normalizer of 5 allows granular scoring

In [13]:
def calculate_completeness_score(evaluation, normalizer=5):
    if not evaluation["is_relevant"]:
        return 0.0
    if evaluation["is_complete"]:
        return 1.0
    # missing_info count capped at normalizer
    missing_count = min(len(evaluation["missing_info"]), normalizer)
    return 1.0 - (missing_count / normalizer)

## Define Entity Completeness Evaluation Prompt

**Goal**: Assess relevance and completeness of extracted entities

**Evaluation criteria**:
- **Relevance**: Does extracted value match the entity name?
- **Completeness**: Does it capture all available information from text?
- Note: Missing Chinese translation alone doesn't mark entity incomplete

**Output format**: JSON with `is_relevant`, `is_complete`, and `missing_info` fields
- Structured observations enable heuristic scoring downstream

In [14]:
system_entity_completeness_prompt = """
You are an expert document understanding AI model. Your task is to evaluate the relevance and completeness of an extracted entity from a given text.

### GOAL
Your goal is to determine if the extracted entity is relevant and if it captures all the information present in the text for that entity.

### INPUT
You will be provided with:
- The text extracted from a document (OCR text).
- An extracted entity with its entity name and value.

### INSTRUCTIONS
- Read the provided text very carefully.
- For the extracted entity, answer:
  1. Is it relevant to the entity name?
  2. Is it complete, i.e., does it capture all relevant information for this entity from the text? If only the chinese translation is missing, still consider it complete.
- If it is incomplete or missing details, list the missing information.
- Do not give a numeric score; only provide structured observations so a numeric score can be computed heuristically later.

### OUTPUT
Provide your evaluation in the following JSON format:
{
  "is_relevant": true_or_false,
  "is_complete": true_or_false,
  "missing_info": ["description_of_missing_information_if_any"]
}
"""

## Instantiate Client for Completeness Evaluation

- Create new Azure OpenAI client instance
- Uses same configuration as detection evaluation
- Ready for completeness assessment prompts

In [15]:
llm_client = get_llm_client_instance(
    azure_endpoint = os.getenv("GPT_4_1_API_ENDPOINT"),
    api_version = os.getenv("GPT_4_1_API_VERSION"),
    azure_deployment = os.getenv("GPT_4_1_API_DEPLOYMENT"),
)

## Evaluate Entity Completeness

- Iterate through all entities in `all_eval`
- **For each entity**:
  - Construct prompt with entity name, value, and OCR text
  - Submit to Azure OpenAI for completeness assessment
  - Parse JSON response and clean markdown artifacts
  - Calculate numeric completeness score using helper function
  - Attach completeness score and missing info to entity record
- Updates `all_eval` in-place with completeness metrics

In [16]:
for item in all_eval:

    messages = [
        {
            "role": "system",
            "content": system_entity_completeness_prompt,
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        f"Entity Name: {item["entity_name"]}\n"
                        f"Entity Value: {item["entity_value"]}\n\n"
                        f"Text:\n{item["adi_text"]}"
                    )
                }
            ]
        }
    ]
    
    response = llm_client.chat.completions.create(
        model=os.getenv("GPT_4_1_API_DEPLOYMENT"),
        messages=messages,
        temperature=0.0,
    )

    eval_completeness = json.loads(
            response.choices[0].message.content
            .replace("```json", "")
            .replace("```", "")
        )
    
    item["entity_completeness_score"] = calculate_completeness_score(eval_completeness)
    item["missing_info"] = eval_completeness.get("missing_info", [])

## Calculate Average Completeness Score

- Sum completeness scores across all entities
- Divide by entity count to get average
- Display overall completeness percentage

In [17]:
total_score = 0.0
num_entities = len(all_eval)

for entity in all_eval:
    total_score += entity["entity_completeness_score"]

avg_completeness_score = total_score / num_entities if num_entities > 0 else 0.0

print(f"Avg Entity Completeness Score across all entities: {100 * avg_completeness_score:.2f}%")


Avg Entity Completeness Score across all entities: 100.00%


## Save Comprehensive Evaluation Results

- Write final `all_eval` data to JSON file
- Includes:
  - Entity detection status
  - Fuzzy similarity scores
  - Completeness scores and missing info
  - Page numbers and entity metadata
- Use `indent=4` for readability and `ensure_ascii=False` for multilingual support
- Output ready for analysis and reporting

In [18]:
all_eval_output_path = "/Users/anishganguli/Documents/Projects/HSBC/PoC/HSBC_IWPB_UW/document_extraction/src/data/results/"

with open(os.path.join(all_eval_output_path, "combined_entity_detection_evaluation.json"), "w") as out_file:
    json.dump(all_eval, out_file, indent=4, ensure_ascii=False)